In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# 1. Lihat 5 baris pertama
print(df.head())

# 2. Cek jumlah baris & kolom
print(df.shape)  # Harusnya (7043, 21)

# 3. Cek tipe data setiap kolom
print(df.dtypes)

# 4. Cek informasi lengkap
print(df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [2]:
# 1. Cek missing values per kolom
print(df.isnull().sum())

# 2. Cek duplikasi data
print(f"Jumlah duplikat: {df.duplicated().sum()}")

# 3. Statistik deskriptif (untuk cek outlier)
print(df.describe())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
Jumlah duplikat: 0
       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


In [3]:
# Ubah TotalCharges dari object ke numeric (paksa error jadi NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Cek lagi tipe datanya
print(df['TotalCharges'].dtypes)

float64


In [4]:
# Cek missing values setelah konversi
print(df.isnull().sum())

# Ada berapa baris yang bermasalah?
print(f"Baris dengan TotalCharges kosong: {df['TotalCharges'].isnull().sum()}")

# Opsi penanganan (pilih salah satu, dokumentasikan keputusanmu):
# Opsi A: Hapus baris yang TotalCharges-nya kosong (karena hanya 11 baris dari 7043)
df_clean = df.dropna(subset=['TotalCharges'])

# Opsi B: Isi dengan median (jika tidak mau hapus data)
# median_charges = df['TotalCharges'].median()
# df['TotalCharges'].fillna(median_charges, inplace=True)

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64
Baris dengan TotalCharges kosong: 11


In [5]:
# Cek outlier di MonthlyCharges pakai IQR
Q1 = df_clean['MonthlyCharges'].quantile(0.25)
Q3 = df_clean['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['MonthlyCharges'] < lower_bound) | (df_clean['MonthlyCharges'] > upper_bound)]
print(f"Jumlah outlier di MonthlyCharges: {len(outliers)}")

# Cek outlier di tenure
Q1_tenure = df_clean['tenure'].quantile(0.25)
Q3_tenure = df_clean['tenure'].quantile(0.75)
IQR_tenure = Q3_tenure - Q1_tenure
outliers_tenure = df_clean[(df_clean['tenure'] < (Q1_tenure - 1.5*IQR_tenure)) | (df_clean['tenure'] > (Q3_tenure + 1.5*IQR_tenure))]
print(f"Jumlah outlier di tenure: {len(outliers_tenure)}")

Jumlah outlier di MonthlyCharges: 0
Jumlah outlier di tenure: 0


In [6]:
# Hapus duplikat
df_clean = df_clean.drop_duplicates()
print(f"Jumlah baris setelah hapus duplikat: {df_clean.shape[0]}")

Jumlah baris setelah hapus duplikat: 7032


In [7]:
# Cek nilai unik di kolom kategorikal
categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                    'PaperlessBilling', 'PaymentMethod', 'Churn']

for col in categorical_cols:
    print(f"{col}: {df_clean[col].unique()}")

gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn: ['No' 'Yes']


In [9]:
# Simpan sebagai CSV baru
df_clean.to_csv('telco_churn_cleaned.csv', index=False)
print("Dataset bersih sudah disimpan!")

Dataset bersih sudah disimpan!


In [10]:
# Cek daftar file di direktori saat ini
import os
print(os.listdir())

['.config', 'telco_churn_cleaned.csv', 'WA_Fn-UseC_-Telco-Customer-Churn.csv', 'sample_data']


In [11]:
# Download file CSV ke laptop
from google.colab import files
files.download('telco_churn_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Buat folder (opsional)
import os
folder = '/content/drive/MyDrive/Tugas_Sains_Data'
os.makedirs(folder, exist_ok=True)

# 3. Simpan CSV
df_clean.to_csv(f'{folder}/telco_churn_cleaned.csv', index=False)
print("✅ File tersimpan di Drive!")

Mounted at /content/drive
✅ File tersimpan di Drive!


In [13]:
# Cek ulang dataset yang tersimpan
df_check = pd.read_csv('/content/drive/MyDrive/Tugas_Sains_Data/telco_churn_cleaned.csv')
print(df_check.info())
print(df_check.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
